In [1]:
!hostname

login.curta.zedat.fu-berlin.de


In [2]:
from photometry.models.lambertian import LambertianModel
from photometry.fitting.least_sq import LeastSquaresFitter
from photometry.core.types import GeometryBatch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path

import os
print(os.getcwd())

/home/kaushim07/photometry_mcmc_env/notebooks


In [3]:
model = LambertianModel()
fitter = LeastSquaresFitter()

In [4]:
project_root = Path.cwd().resolve()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent

data_root = project_root / "data" / "05_aggregated"
phase_order = {"survey": 0, "hamo": 1, "lamo": 2, "rc": 3}

csv_files = sorted(
    data_root.glob("*_phase_curve.csv"),
    key=lambda path: (phase_order.get(path.stem.replace("_phase_curve", "").lower(), 99), path.name),
)
if not csv_files:
    raise FileNotFoundError(f"No csv files found in {data_root}")

phase_groups = {phase: [] for phase in phase_order}
for path in csv_files:
    phase = path.stem.replace("_phase_curve", "").lower()
    if phase in phase_groups:
        phase_groups[phase].append(path)

phase_dfs = {}
phase_best_file = {}
for phase in ["survey", "hamo", "lamo", "rc"]:
    files = phase_groups.get(phase, [])
    if not files:
        phase_dfs[phase] = pd.DataFrame()
        phase_best_file[phase] = None
        continue

    best_path = max(files, key=lambda p: p.stat().st_size)
    phase_best_file[phase] = best_path
    phase_dfs[phase] = pd.read_csv(best_path)

df_survey = phase_dfs["survey"]
df_hamo = phase_dfs["hamo"]
df_lamo = phase_dfs["lamo"]
df_rc = phase_dfs["rc"]


df = df_survey
if df.empty:
    for phase in ["survey", "hamo", "lamo", "rc"]:
        if not phase_dfs[phase].empty:
            df = phase_dfs[phase]
            break

print(f"Project root: {project_root}")
print(f"Aggregated data root: {data_root}")
print(f"Files discovered: {len(csv_files)}")
for phase in ["survey", "hamo", "lamo", "rc"]:
    phase_df = phase_dfs[phase]
    best_file = phase_best_file[phase]
    print(f"{phase.upper()}: files={len(phase_groups[phase])}, rows={len(phase_df):,}")
    if best_file is not None:
        print(f"  representative file: {best_file.name}")

Project root: /home/kaushim07/photometry_mcmc_env
Aggregated data root: /home/kaushim07/photometry_mcmc_env/data/05_aggregated
Files discovered: 4
SURVEY: files=1, rows=90
  representative file: survey_phase_curve.csv
HAMO: files=1, rows=90
  representative file: hamo_phase_curve.csv
LAMO: files=1, rows=70
  representative file: lamo_phase_curve.csv
RC: files=1, rows=61
  representative file: rc_phase_curve.csv


In [7]:
mean_incidence = df_rc["mean_incidence"].to_numpy(dtype=float) * (np.pi / 180.0)
mean_emission = df_rc["mean_emission"].to_numpy(dtype=float) * (np.pi / 180.0)
mean_phase = df_rc["mean_phase"].to_numpy(dtype=float) * (np.pi / 180.0)

geometry_batch = GeometryBatch(
    incidence=mean_incidence,
    emission=mean_emission,
    phase=mean_phase,
)

model = LambertianModel()
fitter = LeastSquaresFitter()

observed = df["median_iof"].to_numpy(dtype=float)

fit_result = fitter.fit(model, geometry_batch, observed)

print("Fitted albedo:", fit_result.fitted_parameters["albedo"])

model.parameters.update(fit_result.fitted_parameters)
predicted = np.asarray(model.reflectance(geometry_batch), dtype=float)

NotImplementedError: LeastSquaresFitter is not implemented yet.

In [6]:
x = df["phase_bin_deg"].to_numpy(dtype=float)
order = np.argsort(x)
x_sorted = x[order]
observed_sorted = observed[order]
predicted_sorted = predicted[order]
yerr_lower = (observed - df["iof_q25"].to_numpy(dtype=float))[order]
yerr_upper = (df["iof_q75"].to_numpy(dtype=float) - observed)[order]
yerr = np.vstack([yerr_lower, yerr_upper])

fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(
    x_sorted,
    observed_sorted,
    yerr=yerr,
    fmt="o",
    markersize=4,
    capsize=3,
    alpha=0.8,
    label="RC data (median I/F, IQR)",
)
ax.plot(
    x_sorted,
    predicted_sorted,
    color="crimson",
    linewidth=2.5,
    label="Lambertian model",
)

ax.set_xlabel("Phase Angle (Degrees)")
ax.set_ylabel("Reflectance (I/F)")
ax.set_title("RC Phase Curve: Lambertian Baseline Fit")
ax.legend()
ax.grid(True, alpha=0.25)

plt.show()

NameError: name 'predicted' is not defined